# 05. Universe Selection & Strategy Construction

## 📋 개요
모델 예측 결과를 기반으로 투자 후보 종목을 선정하고 포트폴리오를 구성합니다.

## ✨ 핵심 프로세스
### 1. 이중 날짜 기준 설정
- **`model_train_date`**: 학습 종료 시점 (참값 존재 → 정확도 평가)
- **`forecast_date`**: 투자 결정 시점 (예측값만 존재)

### 2. 3대 평가 지표
- **정확도 (Accuracy)**: 과거 예측의 RMSE, MAPE 기반
- **수익성 (Return)**: 예측 수익률 (미래 지향)
- **안정성 (Risk)**: 리스크 복합 점수 (작전주, 유동성 등)

### 3. 종합 평가 방법론
**제안: Pareto Optimal Filtering + Weighted Scoring**

```python
# Step 1: Hard Constraints (필수 조건)
universe = universe[
    (universe['accuracy_rank'] <= 500) &  # 정확도 상위 500개
    (universe['risk_composite'] <= 0.7) &  # 리스크 기준
    (universe['is_suspended'] == 0) &       # 거래 가능
    (universe['liquidity_score'] >= min_liquidity)
]

# Step 2: Multi-Objective Scoring
universe['final_score'] = (
    0.40 * universe['accuracy_score'] +    # 40% 가중치
    0.35 * universe['return_score'] +      # 35% 가중치
    0.25 * universe['safety_score']        # 25% 가중치 (역수)
)

# Step 3: Top-K Selection
candidates = universe.nlargest(30, 'final_score')
```

## 🔄 데이터 흐름
```
03단계 예측 결과 (predictions.parquet)
    ↓
[정확도 평가] (model_train_date 기준)
    ↓
04단계 미래 예측 (forecasts.parquet)
    ↓
[수익성 평가] (forecast_date 기준)
    ↓
[리스크 평가] (Universe Meta)
    ↓
[종합 점수 계산]
    ↓
최종 투자 후보 (universe_candidates.parquet)
```

## 🔧 Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings

from src.utils.config import load_config
from src.universe.select_universe import (
    build_production_universe,
    select_candidates_from_scores
)

warnings.filterwarnings('ignore')

## 1️⃣ 설정 및 날짜 기준 정의

In [ ]:
# ==========================================
# 설정 로드
# ==========================================
cfg = load_config()
ref_date = cfg['project']['reference_date']

# ==========================================
# 이중 날짜 기준 설정 ⚠️ 핵심
# ==========================================
MODEL_DATE = cfg['universe']['model_date']  # 예: "2026-01-20" (학습 기준일)
FORECAST_START = pd.to_datetime(MODEL_DATE) + pd.Timedelta(days=1)  # 예측 시작일

print(f"📅 날짜 기준 설정:")
print(f"   - 모델 학습 기준일 (참값 존재): {MODEL_DATE}")
print(f"   - 예측 시작일 (참값 없음): {FORECAST_START.strftime('%Y-%m-%d')}")

# ==========================================
# 경로 설정
# ==========================================
result_dir = Path(cfg['paths']['result_dir']) / ref_date
processed_dir = Path(cfg['paths']['processed_dir']) / ref_date
output_dir = result_dir / 'universe'
output_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📁 경로:")
print(f"   - 예측 결과: {result_dir}")
print(f"   - Universe 출력: {output_dir}")

## 2️⃣ 데이터 로드

In [ ]:
# ==========================================
# 1. 과거 예측 결과 (정확도 평가용)
# ==========================================
print("\n📥 과거 예측 결과 로드 (정확도 평가용)...")
df_past_pred = pd.read_parquet(result_dir / 'predictions.parquet')
df_past_pred['date'] = pd.to_datetime(df_past_pred['date'])

# 학습 기간 데이터만 필터링 (참값 존재)
df_past_pred = df_past_pred[df_past_pred['date'] <= MODEL_DATE].copy()

print(f"   - 총 행수: {len(df_past_pred):,}")
print(f"   - 기간: {df_past_pred['date'].min()} ~ {df_past_pred['date'].max()}")
print(f"   - 종목 수: {df_past_pred['ticker'].nunique()}")

# ==========================================
# 2. 미래 예측 결과 (수익성 평가용)
# ==========================================
print("\n📥 미래 예측 결과 로드 (수익성 평가용)...")
forecast_path = result_dir / 'forecasts' / 'future_forecasts.parquet'

if not forecast_path.exists():
    raise FileNotFoundError(
        f"❌ 미래 예측 파일을 찾을 수 없습니다: {forecast_path}\n"
        "💡 Tip: 04_forecast_future.ipynb를 먼저 실행하세요."
    )

df_future = pd.read_parquet(forecast_path)
df_future['date'] = pd.to_datetime(df_future['date'])

print(f"   - 총 행수: {len(df_future):,}")
print(f"   - 기간: {df_future['date'].min()} ~ {df_future['date'].max()}")
print(f"   - 종목 수: {df_future['ticker'].nunique()}")

# ==========================================
# 3. Feature 데이터셋 (리스크 메타 정보)
# ==========================================
print("\n📥 리스크 메타 데이터 로드...")
df_meta = pd.read_parquet(processed_dir / 'dataset.parquet')
df_meta['date'] = pd.to_datetime(df_meta['date'])

# 최신 날짜의 메타 정보만 사용
latest_meta_date = df_meta['date'].max()
df_meta_latest = df_meta[df_meta['date'] == latest_meta_date].copy()

print(f"   - 기준일: {latest_meta_date}")
print(f"   - 종목 수: {df_meta_latest['ticker'].nunique()}")

## 3️⃣ 정확도 평가 (Accuracy Scoring)

### 🔬 두 가지 방법론 구현

#### Method 1: RMSE 기반 정규화 (기존)
- 장점: 오차 크기를 직접 반영, 안정적
- 단점: 방향성 무시 (예: -5% 예측했는데 +5% 상승)

#### Method 2: 방향성 정확도 + 신뢰도 (신규)
- 장점: "오르락내리락 방향"을 맞추는 능력 평가
- 단점: 극단값 처리 필요

**전략**: 두 방법을 모두 계산하여 최종 단계에서 선택 가능하게 구현

In [ ]:
# ==========================================
# 종목별 예측 정확도 평가 (통합 버전)
# ==========================================
print("\n🎯 정확도 평가 중...")
print("   - Method 1: RMSE 기반 (균형)")
print("   - Method 2: 방향성 정확도 (확실성)")

accuracy_metrics = []

for ticker in tqdm(df_past_pred['ticker'].unique(), desc="정확도 계산"):
    ticker_data = df_past_pred[df_past_pred['ticker'] == ticker].copy()
    
    # 모든 Horizon의 예측 결과 통합
    all_errors = []
    direction_matches = []  # 방향성 일치 여부
    
    for h in range(1, 6):  # h1~h5
        pred_col = f'pred_target_log_close_h{h}'
        true_col = f'true_target_log_close_h{h}'
        
        if pred_col in ticker_data.columns and true_col in ticker_data.columns:
            valid_rows = ticker_data[[pred_col, true_col]].dropna()
            
            if len(valid_rows) > 0:
                # Method 1: 오차 (RMSE용)
                errors = valid_rows[pred_col] - valid_rows[true_col]
                all_errors.extend(errors.values)
                
                # Method 2: 방향성 일치 (DA용)
                # 이전 시점 대비 상승/하락 방향이 맞았는지
                pred_change = valid_rows[pred_col].diff()
                true_change = valid_rows[true_col].diff()
                direction_match = (pred_change * true_change) > 0
                direction_matches.extend(direction_match.dropna().values)
    
    if len(all_errors) > 10:  # 최소 10개 이상 예측값 필요
        all_errors = np.array(all_errors)
        
        # Method 1: RMSE
        rmse = np.sqrt(np.mean(all_errors ** 2))
        mae = np.mean(np.abs(all_errors))
        
        # Method 2: Directional Accuracy
        if len(direction_matches) > 0:
            directional_accuracy = np.mean(direction_matches)
        else:
            directional_accuracy = 0.5  # 기본값 (랜덤 수준)
        
        # Method 2-2: RMSE 역수 기반 신뢰도
        confidence_rmse = 1 / (1 + rmse)
        
        accuracy_metrics.append({
            'ticker': ticker,
            'rmse': rmse,
            'mae': mae,
            'directional_accuracy': directional_accuracy,
            'confidence_rmse': confidence_rmse,
            'num_predictions': len(all_errors)
        })

df_accuracy = pd.DataFrame(accuracy_metrics)

# ==========================================
# Method 1: RMSE 기반 정규화 점수
# ==========================================
# Min-Max Scaling → [0, 1] 범위, 1에 가까울수록 정확
df_accuracy['accuracy_score_v1'] = 1 - (
    (df_accuracy['rmse'] - df_accuracy['rmse'].min()) /
    (df_accuracy['rmse'].max() - df_accuracy['rmse'].min())
)

# ==========================================
# Method 2: 방향성 정확도 점수
# ==========================================
# 0.5(랜덤) ~ 1.0(완벽) 범위를 0~1로 재조정
df_accuracy['accuracy_score_v2'] = (
    (df_accuracy['directional_accuracy'] - 0.5) * 2
).clip(0, 1)  # 혹시 0.5 미만이면 0으로

# ==========================================
# Method 2-2: RMSE 역수 기반 (대안)
# ==========================================
df_accuracy['accuracy_score_v3'] = df_accuracy['confidence_rmse']

# 순위 부여
df_accuracy['accuracy_rank'] = df_accuracy['rmse'].rank()

print(f"\n✅ 정확도 평가 완료")
print(f"   - 평가 종목 수: {len(df_accuracy)}")
print(f"\n[Method 1 - RMSE]")
print(f"   - 평균 RMSE: {df_accuracy['rmse'].mean():.4f}")
print(f"   - 중앙값 RMSE: {df_accuracy['rmse'].median():.4f}")
print(f"\n[Method 2 - Directional Accuracy]")
print(f"   - 평균 방향성 정확도: {df_accuracy['directional_accuracy'].mean():.2%}")
print(f"   - 중앙값: {df_accuracy['directional_accuracy'].median():.2%}")

display(df_accuracy[[
    'ticker', 'rmse', 'directional_accuracy',
    'accuracy_score_v1', 'accuracy_score_v2', 'accuracy_score_v3'
]].head(10))

## 4️⃣ 수익성 평가 (Return Scoring)

### 📐 시간당 로그 수익률 (Time-Normalized Log Return)

**핵심 수식**:
```
일평균 로그 수익률 = [log(매도가) - log(매수가)] / 보유기간
```

**이론적 배경**:
- 주가는 기하 브라운 운동: `dX/dt = rX`
- 로그 수익률의 시간 가산성 (Log Returns are Time-Additive)
- 자본 효율성: 같은 수익이면 짧은 기간이 유리

**제약 조건**:
- **최소 보유 기간**: 5일 (단기 매매 지양)
- **탐색 범위**: 전체 예측 기간 (완전 탐색)

**알고리즘**: 
- NumPy 벡터화 완전 탐색 O(N²)
- 60일 데이터 기준 < 1초

**Why 로그 종가?**
- 모델 타겟이 `target_log_close`이므로 예측값도 로그 스케일
- 불필요한 `np.log()` 변환 제거 (이미 변환됨)

In [ ]:
# ==========================================
# Trading 유틸리티 임포트
# ==========================================
from src.utils.trading import find_best_trade_vectorized

print("✅ Trading 유틸리티 로드 완료")

In [ ]:
# ==========================================
# 종목별 최적 수익률 계산
# ==========================================
print("\n💰 수익성 평가 중 (시간당 로그 수익률 기준)...")

MIN_HOLD_DAYS = 5  # 최소 보유 기간 (1주일)

return_metrics = []
failed_tickers = []

for ticker in tqdm(df_future['ticker'].unique(), desc="최적 수익률 계산"):
    try:
        # 종목별 예측 데이터 추출 (날짜 정렬)
        ticker_data = df_future[
            df_future['ticker'] == ticker
        ].sort_values('date').reset_index(drop=True)
        
        if len(ticker_data) < MIN_HOLD_DAYS:
            failed_tickers.append((ticker, f"데이터 부족 ({len(ticker_data)}일)"))
            continue
        
        # ⭐ 핵심: pred_log_close 직접 사용 (이미 로그 변환됨)
        log_prices = ticker_data['pred_log_close'].values
        
        # 최적 매매 시점 탐색
        buy_idx, sell_idx, daily_log_return, hold_days = find_best_trade_vectorized(
            log_prices, min_hold=MIN_HOLD_DAYS
        )
        
        if np.isnan(daily_log_return) or np.isinf(daily_log_return):
            failed_tickers.append((ticker, "유효한 거래 없음"))
            continue
        
        # 매수/매도 시점 정보
        buy_date = ticker_data.iloc[buy_idx]['date']
        sell_date = ticker_data.iloc[sell_idx]['date']
        buy_price = ticker_data.iloc[buy_idx]['pred_close']
        sell_price = ticker_data.iloc[sell_idx]['pred_close']
        
        # 총 수익률 (검증용)
        total_log_return = log_prices[sell_idx] - log_prices[buy_idx]
        total_return_pct = (np.exp(total_log_return) - 1) * 100
        
        # 연율화 수익률 (참고용, 252 영업일 기준)
        annualized_return = daily_log_return * 252
        
        return_metrics.append({
            'ticker': ticker,
            'daily_log_return': daily_log_return,  # ⭐ 핵심 지표
            'total_log_return': total_log_return,
            'total_return_pct': total_return_pct,
            'annualized_return': annualized_return,
            'hold_days': hold_days,
            'buy_date': buy_date,
            'sell_date': sell_date,
            'buy_price': buy_price,
            'sell_price': sell_price,
            'price_change_pct': (sell_price / buy_price - 1) * 100
        })
    
    except Exception as e:
        failed_tickers.append((ticker, f"오류: {str(e)}"))
        continue

df_return = pd.DataFrame(return_metrics)

# ==========================================
# 점수화 (Min-Max 정규화)
# ==========================================

# [0, 1] 정규화
min_return = df_return['daily_log_return'].min()
max_return = df_return['daily_log_return'].max()

df_return['return_score'] = (
    (df_return['daily_log_return'] - min_return) / (max_return - min_return)
)

# 순위
df_return['return_rank'] = df_return['daily_log_return'].rank(
    ascending=False, method='min'
)

# ==========================================
# 결과 요약
# ==========================================
print(f"\n{'='*65}")
print("✅ 수익성 평가 완료")
print(f"{'='*65}")
print(f"   - 성공 종목 수: {len(df_return):,}")
print(f"   - 실패 종목 수: {len(failed_tickers):,}")

print(f"\n[시간당 로그 수익률 통계]")
print(f"   - 평균: {df_return['daily_log_return'].mean():.6f}")
print(f"   - 중앙값: {df_return['daily_log_return'].median():.6f}")
print(f"   - 표준편차: {df_return['daily_log_return'].std():.6f}")
print(f"   - 최소값: {df_return['daily_log_return'].min():.6f}")
print(f"   - 최대값: {df_return['daily_log_return'].max():.6f}")

print(f"\n[총 수익률 통계]")
print(f"   - 평균: {df_return['total_return_pct'].mean():.2f}%")
print(f"   - 중앙값: {df_return['total_return_pct'].median():.2f}%")

print(f"\n[보유 기간 통계]")
print(f"   - 평균: {df_return['hold_days'].mean():.1f}일")
print(f"   - 중앙값: {df_return['hold_days'].median():.0f}일")

# 수익률 분포
positive_returns = (df_return['total_return_pct'] > 0).sum()
print(f"\n[수익률 분포]")
print(f"   - 양수 수익: {positive_returns}/{len(df_return)} "
      f"({positive_returns/len(df_return)*100:.1f}%)")

print(f"\n[Top 10 종목 - 시간당 로그 수익률 기준]")
display(df_return.nlargest(10, 'daily_log_return')[[
    'ticker', 'return_rank', 'daily_log_return', 'total_return_pct',
    'hold_days', 'buy_date', 'sell_date'
]])

# 실패 종목 확인
if len(failed_tickers) > 0:
    print(f"\n⚠️  실패 종목 샘플 (처음 5개):")
    for ticker, reason in failed_tickers[:5]:
        print(f"   - {ticker}: {reason}")

## 5️⃣ 위험도 평가 (Risk Scoring)

### 📐 종목 내재 위험 (Aleatoric Uncertainty)

**정의**: 모델 예측 오차와 무관한, 종목 자체의 불확정성

**vs 정확도**: 
- **정확도(Accuracy)**: "모델이 얼마나 잘 맞추는가?" (Epistemic)
- **위험도(Risk)**: "이 종목 자체가 얼마나 예측 불가능한가?" (Aleatoric)

### 🔍 5대 표준 지표

| 지표 | 의미 | 해석 |
|------|------|------|
| **Volatility** | 변동성 (기본) | 높을수록 불안정 |
| **Downside Risk** | 하방 위험 | 손실만 측정 (상승은 OK) |
| **VaR / CVaR** | 극단 리스크 | 최악 5% 평균 손실 |
| **Max Drawdown** | 최대 낙폭 | 고점 대비 최대 손실률 |
| **Skew / Kurt** | 분포 형태 | 비대칭성, Fat Tail |

**데이터 소스**: 예측값 시계열 (`pred_log_close`)  
→ "미래 위험도" 평가

**Why 예측값?**
- 과거 위험 ≠ 미래 위험 (시장 국면 변화)
- 모델이 예측한 미래 궤적의 불확정성 측정

In [ ]:
# ==========================================
# Risk 유틸리티 임포트
# ==========================================
from src.utils.risk import (
    calculate_risk_metrics,
    calculate_composite_risk_score,
    normalize_risk_scores
)

print("✅ Risk 유틸리티 로드 완료")

In [ ]:
# ==========================================
# 종목별 위험 지표 계산
# ==========================================
print("\n⚠️  위험도 평가 중 (5대 표준 지표)...")

CONFIDENCE_LEVEL = 0.95  # VaR/CVaR 신뢰수준

risk_results = []
failed_risk_tickers = []

for ticker in tqdm(df_future['ticker'].unique(), desc="위험 지표 계산"):
    try:
        # 종목별 예측 데이터 추출 (날짜 정렬)
        ticker_data = df_future[
            df_future['ticker'] == ticker
        ].sort_values('date').reset_index(drop=True)
        
        if len(ticker_data) < 5:
            failed_risk_tickers.append((ticker, f"데이터 부족 ({len(ticker_data)}일)"))
            continue
        
        # ⭐ 핵심: 예측값 시계열 사용 (미래 위험도)
        log_prices = ticker_data['pred_log_close'].values
        
        # 5대 위험 지표 계산
        metrics = calculate_risk_metrics(
            log_prices,
            confidence_level=CONFIDENCE_LEVEL
        )
        
        # NaN 체크
        if any(np.isnan(v) for v in metrics.values()):
            failed_risk_tickers.append((ticker, "NaN 발생"))
            continue
        
        # 복합 리스크 스코어 계산
        composite_score = calculate_composite_risk_score(metrics)
        
        # 결과 저장
        result = {'ticker': ticker, **metrics, 'risk_composite_raw': composite_score}
        risk_results.append(result)
    
    except Exception as e:
        failed_risk_tickers.append((ticker, f"오류: {str(e)}"))
        continue

df_risk = pd.DataFrame(risk_results)

# ==========================================
# 정규화 및 순위 계산
# ==========================================
df_risk = normalize_risk_scores(df_risk, score_col='risk_composite_raw')

# Safety Score (역순: 높을수록 안전)
df_risk['safety_score'] = 1 - df_risk['risk_score_normalized']

# ==========================================
# 02단계 메타 데이터 병합 (유동성, 거래정지 플래그)
# ==========================================
df_risk = df_risk.merge(
    df_meta_latest[['ticker', 'liquidity_score', 'is_suspended', 'is_delisted']],
    on='ticker',
    how='left'
)

# ==========================================
# 결과 요약
# ==========================================
print(f"\n{'='*65}")
print("✅ 위험도 평가 완료")
print(f"{'='*65}")
print(f"   - 성공 종목 수: {len(df_risk):,}")
print(f"   - 실패 종목 수: {len(failed_risk_tickers):,}")

print(f"\n[1. Volatility (변동성)]")
print(f"   - 평균: {df_risk['volatility'].mean():.6f}")
print(f"   - 중앙값: {df_risk['volatility'].median():.6f}")
print(f"   - 연율화: {df_risk['volatility'].mean() * np.sqrt(252):.2%}")

print(f"\n[2. Downside Risk (하방 위험)]")
print(f"   - 평균: {df_risk['downside_risk'].mean():.6f}")
downside_ratio = df_risk['downside_risk'].mean() / df_risk['volatility'].mean()
print(f"   - Downside/Total 비율: {downside_ratio:.2%}")

print(f"\n[3. VaR/CVaR (극단 리스크, 95% 신뢰)]")
print(f"   - VaR (5th percentile): {df_risk['var'].mean():.6f}")
print(f"   - CVaR (Expected Shortfall): {df_risk['cvar'].mean():.6f}")
print(f"   - CVaR 연율화: {df_risk['cvar'].mean() * 252:.2%}")

print(f"\n[4. Maximum Drawdown (최대 낙폭)]")
print(f"   - 평균 MDD: {df_risk['max_drawdown'].mean():.2%}")
print(f"   - 중앙값 MDD: {df_risk['max_drawdown'].median():.2%}")
print(f"   - 최악 MDD: {df_risk['max_drawdown'].min():.2%}")

print(f"\n[5. Skewness/Kurtosis (분포 형태)]")
avg_skew = df_risk['skewness'].mean()
print(f"   - 평균 Skewness: {avg_skew:.4f} ", end="")
print(f"({'하락 쏠림' if avg_skew < 0 else '상승 쏠림'})")
avg_kurt = df_risk['excess_kurtosis'].mean()
print(f"   - 평균 Excess Kurtosis: {avg_kurt:.4f} ", end="")
print(f"({'Fat Tail' if avg_kurt > 0 else 'Thin Tail'})")

print(f"\n[복합 리스크 스코어]")
print(f"   - 평균: {df_risk['risk_composite_raw'].mean():.6f}")
print(f"   - 정규화 후 평균: {df_risk['risk_score_normalized'].mean():.4f}")

# ==========================================
# 안전한 종목 Top 10 (낮은 위험)
# ==========================================
print(f"\n[Top 10 안전 종목 - 낮은 위험 기준]")
display(df_risk.nsmallest(10, 'risk_composite_raw')[[
    'ticker', 'risk_rank', 'volatility', 'downside_risk',
    'cvar', 'max_drawdown', 'excess_kurtosis', 'safety_score'
]])

# ==========================================
# 위험한 종목 Top 10 (높은 위험)
# ==========================================
print(f"\n[Top 10 위험 종목 - 높은 위험 기준]")
display(df_risk.nlargest(10, 'risk_composite_raw')[[
    'ticker', 'risk_rank', 'volatility', 'downside_risk',
    'cvar', 'max_drawdown', 'excess_kurtosis', 'safety_score'
]])

# 실패 종목 샘플
if len(failed_risk_tickers) > 0:
    print(f"\n⚠️  실패 종목 샘플 (처음 5개):")
    for ticker, reason in failed_risk_tickers[:5]:
        print(f"   - {ticker}: {reason}")

In [ ]:
# ==========================================
# 위험 지표 간 상관관계 분석 (선택)
# ==========================================
print("\n[위험 지표 간 상관관계]")

risk_cols = ['volatility', 'downside_risk', 'var', 'cvar', 'max_drawdown', 'excess_kurtosis']
correlation = df_risk[risk_cols].corr()

display(correlation.round(3))

# 해석 가이드
print("\n💡 해석 가이드:")
print("   - Volatility vs Downside Risk: 높으면 손실 변동성 우세")
print("   - CVaR vs MDD: 높으면 극단 리스크 연관성 강함")
print("   - Excess Kurtosis vs 타 지표: 높으면 Fat Tail이 주요 위험 요인")

## 5-1️⃣ 하드 필터링 (Hard Filters)

### 🚧 한국 시장 특수성 반영

**목적**: 투자 불가능하거나 위험한 종목을 **물리적으로 제거**

| 필터 | 제거 대상 | 기준 |
|------|-----------|------|
| **1. 거래정지/상폐** | `is_suspended=1`, `is_delisted=1` | 매매 불가능 |
| **2. 작전주/테마주** | 20일 +100% 급등 + 거래량 5배 폭증 | Phase 2 설계서 2-1-2 |
| **3. 저가주** | 평균 예측가 < 1,000원 | 극심한 변동성, 작전 가능성 |
| **4. 저유동성** | 20일 평균 거래대금 < 5천만 원 | 매매 체결 불가 위험 |

**철학**: 
- "안 좋은 종목"이 아니라 "투자 불가능한 종목" 제거
- 스코어링 이전 단계 (0점 vs 제외의 차이)

**Phase 2 설계서 인용**:
> "학습은 전부, 투자 후보는 엄선해서"

In [ ]:
# ==========================================
# Filter 유틸리티 임포트
# ==========================================
from src.utils.filters import (
    apply_hard_filters,
    analyze_filter_impact
)

print("✅ Hard Filter 유틸리티 로드 완료")

In [ ]:
# ==========================================
# 전체 평가 데이터 통합 준비
# ==========================================
print("\n🔗 평가 데이터 통합 중...")

# 1. 정확도 평가 결과 (3가지 방법)
df_eval = df_accuracy_combined.copy()

# 2. 수익성 평가 결과 병합
df_eval = df_eval.merge(
    df_profitability[[
        'ticker', 'daily_log_return', 'total_return_pct',
        'hold_days', 'buy_date', 'sell_date'
    ]],
    on='ticker',
    how='inner'
)

# 3. 위험도 평가 결과 병합
df_eval = df_eval.merge(
    df_risk[[
        'ticker', 'volatility', 'downside_risk', 'cvar_95',
        'max_drawdown', 'skewness', 'kurtosis',
        'risk_composite_raw', 'risk_score_normalized', 'safety_score'
    ]],
    on='ticker',
    how='inner'
)

print(f"✅ 통합 완료: {len(df_eval):,}개 종목")
print(f"   - 정확도 지표: {len([c for c in df_eval.columns if 'accuracy' in c or 'error' in c])}개")
print(f"   - 수익성 지표: 6개")
print(f"   - 위험 지표: 8개")

In [ ]:
# ==========================================
# 하드 필터링 적용
# ==========================================

# 필터링 설정
filter_config = {
    'min_liquidity': 50_000_000,      # 5천만 원 (소액 계좌 기준)
    'min_price': 1000.0,              # 1,000원 (동전주 배제)
    'surge_threshold': 1.0,           # 100% 급등 (작전주 의심)
    'volume_multiplier': 5.0          # 5배 거래량 폭증
}

# 통합 필터 적용
df_filtered, filter_stats = apply_hard_filters(
    df=df_eval,
    df_future=df_future,
    meta_df=df_meta_latest,
    config=filter_config,
    verbose=True
)

# ==========================================
# 필터링 영향 분석
# ==========================================
print("\n📊 필터링 영향 분석")
df_filter_report = analyze_filter_impact(filter_stats)
display(df_filter_report)

In [ ]:
# ==========================================
# 필터링 전후 비교
# ==========================================
print("\n" + "=" * 65)
print("📊 필터링 전후 비교")
print("=" * 65)

# 기본 통계
print(f"\n[종목 수]")
print(f"   - 필터링 전: {len(df_eval):,}개")
print(f"   - 필터링 후: {len(df_filtered):,}개")
print(f"   - 제거 비율: {(1 - len(df_filtered)/len(df_eval)) * 100:.1f}%")

# 수익성 비교
print(f"\n[평균 기대 수익률]")
print(f"   - 필터링 전: {df_eval['daily_log_return'].mean():.6f} (일평균)")
print(f"   - 필터링 후: {df_filtered['daily_log_return'].mean():.6f} (일평균)")
improvement = (
    df_filtered['daily_log_return'].mean() - df_eval['daily_log_return'].mean()
) * 252 * 100
print(f"   - 개선 효과: {improvement:+.2f}%p (연율)")

# 위험도 비교
print(f"\n[평균 위험 점수]")
print(f"   - 필터링 전: {df_eval['risk_composite_raw'].mean():.6f}")
print(f"   - 필터링 후: {df_filtered['risk_composite_raw'].mean():.6f}")
risk_reduction = (
    (df_eval['risk_composite_raw'].mean() - df_filtered['risk_composite_raw'].mean())
    / df_eval['risk_composite_raw'].mean() * 100
)
print(f"   - 위험 감소: {risk_reduction:.1f}%")

# 정확도 비교
print(f"\n[평균 정확도]")
print(f"   - 필터링 전: {df_eval['accuracy_combined'].mean():.4f}")
print(f"   - 필터링 후: {df_filtered['accuracy_combined'].mean():.4f}")

print("\n" + "=" * 65)

## 6️⃣ 수익률 기준 정렬 및 넓은 Top-K 선정

In [ ]:
print("\n" + "="*65)
print("6️⃣ 수익률 기준 Universe 선정")
print("="*65)

# ==========================================
# 1. 3대 지표 통합
# ==========================================
print("\n🔗 평가 지표 통합 중...")

df_universe = df_accuracy.merge(df_return, on='ticker', how='inner')
df_universe = df_universe.merge(df_risk, on='ticker', how='inner')

print(f"   - 통합 완료: {len(df_universe):,}개 종목")

# ==========================================
# 2. Hard Constraints (필수 조건)
# ==========================================
print("\n🚫 Hard Constraints 적용 중...")

initial_count = len(df_universe)

# 거래 가능 종목만
df_universe = df_universe[
    (df_universe['is_suspended'] == 0) &
    (df_universe['is_delisted'] == 0)
]

# 최소 유동성 기준
MIN_LIQUIDITY = 5e7  # 5천만 원
df_universe = df_universe[df_universe['liquidity_score'] >= MIN_LIQUIDITY]

# 리스크 상한 (선택: 너무 위험한 종목 제거)
MAX_RISK = 0.8  # 기존 0.7보다 완화 (더 많은 후보 확보)
df_universe = df_universe[df_universe['risk_composite'] <= MAX_RISK]

# 정확도 하한 (선택: 너무 부정확한 종목 제거)
# 기존 상위 500개 → 상위 1000개로 완화
df_universe = df_universe[df_universe['accuracy_rank'] <= 1000]

print(f"   - 필터링 전: {initial_count:,}개")
print(f"   - 필터링 후: {len(df_universe):,}개")
print(f"   - 제거됨: {initial_count - len(df_universe):,}개")

# ==========================================
# 3. 수익률 기준 내림차순 정렬
# ==========================================
print("\n📊 수익률 기준 정렬 중...")

# 핵심 지표: daily_log_return (시간당 로그 수익률)
df_universe = df_universe.sort_values('daily_log_return', ascending=False).reset_index(drop=True)

# 순위 부여
df_universe['return_rank'] = range(1, len(df_universe) + 1)

print(f"   - 정렬 기준: daily_log_return (시간당 로그 수익률)")
print(f"   - 1위 수익률: {df_universe.iloc[0]['daily_log_return']:.6f}")
print(f"   - 중앙값 수익률: {df_universe['daily_log_return'].median():.6f}")

# ==========================================
# 4. 넓은 Top-K 선정
# ==========================================
TOP_K = 200  # 충분히 넓게 (사용자가 직접 선택할 여지)

if len(df_universe) < TOP_K:
    print(f"\n⚠️  필터링 후 종목 수({len(df_universe)})가 TOP_K({TOP_K})보다 적습니다.")
    print(f"   → 전체 {len(df_universe)}개 종목을 후보로 선정합니다.")
    df_candidates = df_universe.copy()
else:
    df_candidates = df_universe.head(TOP_K).copy()

print(f"\n✅ 최종 후보 선정 완료")
print(f"   - 후보 종목 수: {len(df_candidates):,}개")
print(f"   - 선정 기준: 예상 수익률 상위 {TOP_K}개")

# ==========================================
# 5. 후보군 통계
# ==========================================
print(f"\n📊 후보군 통계:")
print(f"   - 평균 일평균 로그 수익률: {df_candidates['daily_log_return'].mean():.6f}")
print(f"   - 평균 총 수익률: {df_candidates['total_return_pct'].mean():.2f}%")
print(f"   - 평균 보유 기간: {df_candidates['hold_days'].mean():.1f}일")
print(f"   - 평균 방향성 정확도: {df_candidates['directional_accuracy'].mean():.2%}")
print(f"   - 평균 RMSE: {df_candidates['rmse'].mean():.4f}")
print(f"   - 평균 리스크 점수: {df_candidates['risk_composite'].mean():.3f}")

# 수익률 분포
positive_returns = (df_candidates['total_return_pct'] > 0).sum()
print(f"\n   - 양수 수익 종목: {positive_returns}/{len(df_candidates)} ({positive_returns/len(df_candidates)*100:.1f}%)")

## 7️⃣ 사용자 선택을 위한 상세 리포트

In [ ]:
print("\n" + "="*65)
print("7️⃣ 투자 후보 상세 리포트 생성")
print("="*65)

# ==========================================
# 1. ticker_name_map 로드 (종목명 표시)
# ==========================================
try:
    master_path = Path(cfg['paths']['raw_dir']) / ref_date / f"ticker_master_{ref_date}.csv"
    df_master = pd.read_csv(master_path)
    ticker_name_map = dict(zip(df_master['ticker'].astype(str), df_master['name']))
    df_candidates['종목명'] = df_candidates['ticker'].map(ticker_name_map)
    print("✅ ticker_master 로드 완료")
except Exception as e:
    print(f"⚠️  ticker_master 로드 실패: {e}")
    df_candidates['종목명'] = df_candidates['ticker']

# ==========================================
# 2. 리포트용 컬럼 정리 (사용자 의사결정 지원)
# ==========================================

# 가독성을 위한 컬럼명 변경
df_candidates_report = df_candidates.copy()

df_candidates_report['순위'] = df_candidates_report['return_rank']
df_candidates_report['티커'] = df_candidates_report['ticker']

# 수익성 지표
df_candidates_report['예상일평균수익률(로그)'] = df_candidates_report['daily_log_return']
df_candidates_report['예상총수익률(%)'] = df_candidates_report['total_return_pct']
df_candidates_report['최적보유기간(일)'] = df_candidates_report['hold_days']

# 정확도 지표
df_candidates_report['방향성정확도(%)'] = (df_candidates_report['directional_accuracy'] * 100).round(2)
df_candidates_report['RMSE'] = df_candidates_report['rmse'].round(4)

# 위험 지표
df_candidates_report['리스크점수'] = df_candidates_report['risk_composite'].round(3)
df_candidates_report['변동성'] = df_candidates_report['volatility'].round(4)
df_candidates_report['최대낙폭(%)'] = (df_candidates_report['max_drawdown'] * 100).round(2)

# 매매 정보
df_candidates_report['매수일'] = df_candidates_report['buy_date']
df_candidates_report['매도일'] = df_candidates_report['sell_date']
df_candidates_report['매수가'] = df_candidates_report['buy_price'].round(0)
df_candidates_report['매도가'] = df_candidates_report['sell_price'].round(0)

# 메타 정보
df_candidates_report['유동성점수'] = df_candidates_report['liquidity_score']

# ==========================================
# 3. 최종 리포트 컬럼 순서
# ==========================================
report_cols = [
    '순위', '티커', '종목명',
    
    # 수익성 (주요 지표)
    '예상일평균수익률(로그)', '예상총수익률(%)', '최적보유기간(일)',
    
    # 정확도
    '방향성정확도(%)', 'RMSE',
    
    # 위험
    '리스크점수', '변동성', '최대낙폭(%)',
    
    # 매매 정보
    '매수일', '매도일', '매수가', '매도가',
    
    # 메타
    '유동성점수'
]

df_report = df_candidates_report[report_cols]

# ==========================================
# 4. Top 20 미리보기
# ==========================================
print("\n📊 Top 20 종목 미리보기:")
print("\n[주요 지표 설명]")
print("   - 예상일평균수익률(로그): 시간당 복리 수익률 (높을수록 자본 효율 우수)")
print("   - 예상총수익률(%): 매수~매도 총 수익률")
print("   - 방향성정확도(%): 과거 예측의 상승/하락 방향 적중률")
print("   - 리스크점수: 복합 위험 지표 (0~1, 낮을수록 안전)")
print("   - 최적보유기간: 최대 수익을 위한 권장 보유일\n")

# 핵심 컬럼만 표시 (가독성)
display_cols_short = [
    '순위', '종목명', '예상총수익률(%)', '최적보유기간(일)',
    '방향성정확도(%)', '리스크점수', '매수가', '매도가'
]

display(df_report[display_cols_short].head(20))

## 8️⃣ 결과 저장

In [ ]:
print("\n" + "="*65)
print("8️⃣ 결과 저장")
print("="*65)

# ==========================================
# 1. 전체 Universe (평가 완료)
# ==========================================
full_universe_path = output_dir / 'universe_full.parquet'
df_universe.to_parquet(full_universe_path, index=False)
print(f"\n💾 전체 Universe 저장: {full_universe_path}")
print(f"   - 종목 수: {len(df_universe):,}")

# ==========================================
# 2. Top-K 후보 (Parquet)
# ==========================================
candidates_path = output_dir / 'universe_candidates.parquet'
df_candidates.to_parquet(candidates_path, index=False)
print(f"\n💾 투자 후보 저장: {candidates_path}")
print(f"   - 종목 수: {len(df_candidates):,}")

# ==========================================
# 3. 상세 리포트 (CSV, 사람 가독성 우선)
# ==========================================
report_path = output_dir / 'investment_report.csv'
df_report.to_csv(report_path, index=False, encoding='utf-8-sig')
print(f"\n💾 상세 리포트 저장: {report_path}")
print(f"   - 형식: CSV (Excel 호환)")
print(f"   - 컬럼 수: {len(report_cols)}개")

# ==========================================
# 4. Excel용 요약 시트 (선택)
# ==========================================
try:
    excel_path = output_dir / 'investment_report.xlsx'
    
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        # Sheet 1: Top 20 요약
        df_report.head(20).to_excel(writer, sheet_name='Top20', index=False)
        
        # Sheet 2: 전체 후보
        df_report.to_excel(writer, sheet_name='전체후보', index=False)
        
        # Sheet 3: 위험별 분류
        df_risk_groups = df_report.copy()
        df_risk_groups['위험등급'] = pd.cut(
            df_risk_groups['리스크점수'],
            bins=[0, 0.3, 0.5, 0.7, 1.0],
            labels=['낮음', '보통', '높음', '매우높음']
        )
        
        for risk_level in ['낮음', '보통', '높음', '매우높음']:
            df_level = df_risk_groups[df_risk_groups['위험등급'] == risk_level]
            if len(df_level) > 0:
                df_level.to_excel(writer, sheet_name=f'위험_{risk_level}', index=False)
    
    print(f"\n💾 Excel 리포트 저장: {excel_path}")
    print(f"   - Sheet: Top20, 전체후보, 위험_낮음, 위험_보통 등")
    
except Exception as e:
    print(f"\n⚠️  Excel 저장 실패 (openpyxl 필요): {e}")

# ==========================================
# 5. 필터링 통계 저장
# ==========================================
filter_stats_path = output_dir / 'filter_statistics.json'

filter_summary = {
    'initial_count': initial_count,
    'final_count': len(df_candidates),
    'removal_count': initial_count - len(df_candidates),
    'removal_rate_%': (initial_count - len(df_candidates)) / initial_count * 100,
    
    'constraints': {
        'min_liquidity': MIN_LIQUIDITY,
        'max_risk': MAX_RISK,
        'max_accuracy_rank': 1000
    },
    
    'statistics': {
        'avg_expected_return_%': df_candidates['total_return_pct'].mean(),
        'avg_hold_days': df_candidates['hold_days'].mean(),
        'avg_directional_accuracy_%': df_candidates['directional_accuracy'].mean() * 100,
        'avg_rmse': df_candidates['rmse'].mean(),
        'avg_risk_score': df_candidates['risk_composite'].mean()
    }
}

import json
with open(filter_stats_path, 'w', encoding='utf-8') as f:
    json.dump(filter_summary, f, indent=2, ensure_ascii=False)

print(f"\n💾 필터링 통계 저장: {filter_stats_path}")

print("\n" + "="*65)
print("✅ [Step 5] Universe 선정 완료")
print("="*65)
print(f"\n💡 다음 단계:")
print(f"   1. Excel/CSV 파일 열기: {report_path.name}")
print(f"   2. 수익률, 정확도, 위험을 종합 검토")
print(f"   3. 최종 투자 종목 수동 선택 (권장: 20~30개)")
print(f"   4. 선택한 종목으로 06단계 포트폴리오 최적화 진행")

## 🏁 완료 및 다음 단계

### ✅ 생성된 산출물
- **전체 Universe**: `universe_full.parquet` (모든 평가 완료 종목)
- **투자 후보**: `universe_candidates.parquet` (Top 30)
- **리포트**: `investment_report.csv` (사람 가독성)

### 🔬 구현된 두 가지 전략 비교

#### Strategy A: Multi-Objective Weighted (균형)
```python
score = 0.40 × accuracy + 0.35 × return + 0.25 × safety
```

**장점**:
- ✅ 안정적이고 해석 가능
- ✅ 각 지표가 균형있게 기여
- ✅ 극단값에 강건함

**단점**:
- ⚠️ 정확도 낮은 종목도 수익률로 보상받을 수 있음
- ⚠️ "양치기 소년" 종목 배제 약함

**추천 상황**:
- 보수적 운용
- 데이터가 적거나 불확실할 때
- 다양한 종목에 분산 투자

---

#### Strategy B: Confidence-Weighted (확실성) ⭐ 기본값
```python
score = expected_return × (confidence^1.5)
```

**장점**:
- ✅ "확실한 수익"에 집중 (Kelly Criterion)
- ✅ 방향성 자주 틀리는 종목 자동 배제
- ✅ 직관적 ("신뢰도 없으면 수익률 무의미")

**단점**:
- ⚠️ 극단값에 민감 (신뢰도 0.9 vs 0.7의 차이가 큼)
- ⚠️ 데이터 적으면 불안정

**추천 상황**:
- 공격적 운용
- 데이터가 충분할 때 (학습 기간 6개월+)
- 소수 종목 집중 투자

---

### 🎯 전략 선택 가이드

| 조건 | 권장 전략 | 이유 |
|------|----------|------|
| 학습 데이터 < 3개월 | Strategy A | 신뢰도 추정 불안정 |
| 학습 데이터 ≥ 6개월 | Strategy B | 신뢰도 안정적 |
| 계좌 크기 < $10,000 | Strategy A | 분산 필요 |
| 계좌 크기 ≥ $50,000 | Strategy B | 집중 투자 가능 |
| 시장 변동성 높음 | Strategy A | 안정성 우선 |
| 시장 변동성 낮음 | Strategy B | 수익 극대화 |
| 백테스트 Sharpe < 1.0 | Strategy A | 보수적 접근 |
| 백테스트 Sharpe ≥ 1.5 | Strategy B | 공격적 접근 |

---

### 📊 평가 방법론 요약

#### 1. 이중 날짜 기준
```
model_train_date (2025-02-06)
  ↓ [정확도 평가]
  - RMSE 기반 (Method 1)
  - 방향성 정확도 기반 (Method 2) ⭐
  
forecast_date (2025-02-07~)
  ↓ [수익성 평가]
  - 예측 수익률 (h5 기준)
```

#### 2. 필터링 단계
1. **Hard Constraints** (필수 조건)
   - 거래정지/상장폐지 제외
   - 최소 유동성 5천만 원
   - 리스크 0.7 이하
   - 정확도 상위 500개

2. **Soft Ranking** (점수 기반)
   - Strategy A: 가중 선형 결합
   - Strategy B: 신뢰도 가중 곱셈 ⭐

---

### 🚀 다음 작업
- **06단계**: 포트폴리오 최적화 (Constrained MVO 등)
- **07단계**: 백테스트 및 리스크 시뮬레이션
- **08단계**: 주간 리밸런싱 전략 구현

### 💡 전략 변경 방법
```python
# Strategy A로 변경하려면:
df_universe['final_score'] = df_universe['score_strategy_a']

# Strategy B 변형 (RMSE 역수 기반):
df_universe['final_score'] = df_universe['score_strategy_b_v2']

# 신뢰도 가중치 조정:
CONFIDENCE_POWER = 2.0  # 더 공격적
CONFIDENCE_POWER = 1.0  # 더 보수적
```

### 🔍 백테스트 권장사항
1. 두 전략을 모두 백테스트하여 성과 비교
2. Sharpe Ratio, MDD, Win Rate 측정
3. 시장 국면별(Bull/Bear) 성과 분석
4. 최적 전략 선택 또는 하이브리드 구성

---

**Last Updated**: 2026-02-02  
**Pipeline Step**: 05 (Universe Selection)  
**Method**: Hybrid (Strategy A + Strategy B)